# EHR Timeline Transformer — Colab Training

Run the full pipeline: data prep → baselines → transformer training → test predictions.

**Colab workflow**
1. Upload raw data to Google Drive once (see setup cells below).
2. On code changes: `git push` locally, then re-run the **Setup** cells — they use `git pull`, not a fresh clone.
3. Data and checkpoints stay on Drive across runtime restarts.

**Requirements:** GPU runtime recommended for transformer training.

In [ ]:
# --- Edit these paths for your environment ---
REPO_URL = "https://github.com/YOUR_USERNAME/modeling_patient_timelines.git"
REPO_DIR = "/content/modeling_patient_timelines"

# Persistent locations on Google Drive (upload data once; reused every session)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ehr_project"
DRIVE_DATA_DIR = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_OUTPUTS_DIR = f"{DRIVE_PROJECT_DIR}/outputs"

# Optional: zip with raw data for first-time setup. Set to None after data is extracted.
DATA_ZIP = f"{DRIVE_PROJECT_DIR}/ehr_data.zip"  # or None

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import zipfile

drive.mount("/content/drive")

REQUIRED = [
    "patient_splits.csv",
    "target_conditions.csv",
    "test_anchors.csv",
    "train_val",
    "test",
]

data_dir = Path(DRIVE_DATA_DIR)
missing = [name for name in REQUIRED if not (data_dir / name).exists()]

if missing:
    if DATA_ZIP and Path(DATA_ZIP).exists():
        print(f"First-time setup: extracting {DATA_ZIP}")
        data_dir.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP) as zf:
            names = zf.namelist()
            top_levels = {n.split("/")[0] for n in names if n.strip()}
            if top_levels == {"data"}:
                zf.extractall(data_dir.parent)
            else:
                data_dir.mkdir(parents=True, exist_ok=True)
                zf.extractall(data_dir)
        missing = [name for name in REQUIRED if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing data in {DRIVE_DATA_DIR}: {', '.join(missing)}. "
            "Upload the raw CSV folders to Drive or set DATA_ZIP."
        )
else:
    print(f"Data ready at {DRIVE_DATA_DIR}")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)

if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)
    print(f"Updated existing repo at {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    print(f"Cloned repo to {REPO_DIR}")

subprocess.run(
    ["pip", "install", "-q", "-r", str(repo_dir / "requirements.txt")],
    check=True,
)

# Link persistent Drive folders into the repo (no re-upload on code updates).
for link_name, target in [("data", DRIVE_DATA_DIR), ("outputs", DRIVE_OUTPUTS_DIR)]:
    link = repo_dir / link_name
    target_path = Path(target).resolve()
    target_path.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() and link.resolve() == target_path:
        continue
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        backup = repo_dir / f"{link_name}.colab_backup"
        if backup.exists():
            shutil.rmtree(backup)
        link.rename(backup)
        print(f"Backed up existing {link_name}/ to {backup.name}/")
    link.symlink_to(target_path, target_is_directory=True)
    print(f"Linked {link} -> {target_path}")

os.chdir(repo_dir)
print(f"Working directory: {repo_dir}")

In [ ]:
from pathlib import Path

for name in REQUIRED:
    path = Path("data") / name
    assert path.exists(), f"Missing data/{name}"

print("All required data files are present.")

In [ ]:
!python scripts/prepare_data.py --config configs/transformer.yaml

In [ ]:
!python scripts/train_baseline.py --config configs/baseline.yaml

In [ ]:
# Login to wandb (optional)
# import wandb
# wandb.login()

!python scripts/train_transformer.py --config configs/transformer.yaml

In [ ]:
!python scripts/make_predictions.py --checkpoint outputs/checkpoints/best_model.pt --output outputs/predictions.csv

In [ ]:
from google.colab import files

files.download("outputs/predictions.csv")

In [ ]:
# Optional: upload to Hugging Face Hub
# !python scripts/upload_to_hub.py --repo-id YOUR_USERNAME/ehr-timeline-transformer